# Commands 19A–19C: In-Hospital Decedent Analysis

This notebook evaluates documented inpatient palliative-care use among adult HM hospitalizations ending in in-hospital death. Run all cells to refresh the descriptive and adjusted analyses.

## Methods

Decedents have `DIED=1`; records with missing `DIED` are excluded. Models use the same demographic, socioeconomic, cancer-excluded Charlson, hospital, year, and mutually exclusive subtype definitions as Commands 16–17. Estimates use `DISCWT` and year-specific `NIS_STRATUM`. Because `HOSP_NIS` is unavailable by study decision, discharges are variance units and inference is a strata-adjusted approximation.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks': REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path: sys.path.insert(0, str(REPO_ROOT))
from src.phase_10_decedents import main
summary = main()

{
  "died_missing_records": 441,
  "decedent_unweighted_n": 53737,
  "sepsis_decedent_unweighted_n": 27210,
  "command_19a_p_value": "<0.001",
  "command_19b_interaction_test": {
    "wald_chi_square": 18.816,
    "degrees_of_freedom": 8,
    "p_value": "0.016"
  },
  "command_19c_subtype_test": {
    "wald_chi_square": 20.066,
    "degrees_of_freedom": 8,
    "p_value": "0.010"
  },
  "variance_note": "DISCWT-weighted estimates with year-specific NIS_STRATUM linearization and discharge-level variance units; not full NIS hospital-cluster-adjusted inference."
}


## Command 19A: Palliative-care use among decedents

In [2]:
table_19a = pd.read_csv(REPO_ROOT / 'outputs/phase_10/decedent_palliative_care_by_sepsis.csv', keep_default_na=False)
table_19a.columns = ['Cohort', 'Unweighted sample n', 'Unweighted palliative-care n', 'Weighted hospitalizations', 'Weighted palliative-care n', 'Weighted palliative-care %', '95% CI lower, %', '95% CI upper, %', 'P-value']
display(table_19a.style.hide(axis='index').format(thousands=','))

Cohort,Unweighted sample n,Unweighted palliative-care n,Weighted hospitalizations,Weighted palliative-care n,Weighted palliative-care %,"95% CI lower, %","95% CI upper, %",P-value
Decedents without documented sepsis,"26,527","15,179","132,635","75,895",57.220000,56.630000,57.810000,<0.001
Decedents with documented sepsis,"27,210","13,329","136,050","66,645",48.990000,48.400000,49.570000,
Total decedents,"53,737","28,508","268,685","142,540",53.050000,52.640000,53.460000,<0.001


## Command 19B: Adjusted decedent interaction model

In [3]:
table_19b = pd.read_csv(REPO_ROOT / 'outputs/phase_10/adjusted_decedent_interaction.csv', keep_default_na=False)
table_19b['Difference 95% CI, pp'] = table_19b.apply(lambda row: '—' if row['hm_subtype'] == 'Total' else f"{float(row['difference_ci_95_lower_pp']):.2f}–{float(row['difference_ci_95_upper_pp']):.2f}", axis=1)
table_19b = table_19b[['hm_subtype','adjusted_probability_no_sepsis_percent','adjusted_probability_sepsis_percent','adjusted_difference_pp','Difference 95% CI, pp','subtype_p_value','overall_interaction_p_value']]
table_19b.columns = ['HM subtype','Adjusted probability without sepsis, %','Adjusted probability with sepsis, %','Adjusted difference, pp','Difference 95% CI, pp','Subtype p-value','Overall interaction p-value']
display(table_19b.style.hide(axis='index'))
display(pd.DataFrame([summary['command_19b_interaction_test']]).style.hide(axis='index'))

HM subtype,"Adjusted probability without sepsis, %","Adjusted probability with sepsis, %","Adjusted difference, pp","Difference 95% CI, pp",Subtype p-value,Overall interaction p-value
Lymphoma,59.08,50.57,-8.52,-10.06–-6.97,<0.001,0.016
AML,59.47,50.12,-9.35,-11.52–-7.18,<0.001,
CML,56.39,52.5,-3.89,-9.15–1.38,0.148,
CLL/chronic leukemia,54.75,47.49,-7.26,-9.70–-4.81,<0.001,
ALL/unspecified acute leukemia,55.74,47.3,-8.45,-13.38–-3.52,<0.001,
Other leukemia,54.89,48.89,-6.0,-10.90–-1.10,0.016,
Myeloma/plasma-cell neoplasm,56.66,50.39,-6.27,-8.42–-4.12,<0.001,
MDS,54.57,48.34,-6.24,-8.63–-3.84,<0.001,
MPN,51.64,48.37,-3.27,-5.92–-0.62,0.015,
Total,—,—,—,—,—,0.016


wald_chi_square,degrees_of_freedom,p_value
18.816000,8,0.016


## Command 19C: Sepsis decedents by HM subtype — unadjusted

In [4]:
table_19c_raw = pd.read_csv(REPO_ROOT / 'outputs/phase_10/sepsis_decedent_subtype_unadjusted.csv', keep_default_na=False).drop(columns='cohort')
table_19c_raw.columns = ['HM subtype','Unweighted sample n','Unweighted palliative-care n','Weighted hospitalizations','Weighted palliative-care n','Weighted palliative-care %','95% CI lower, %','95% CI upper, %','Overall p-value']
display(table_19c_raw.style.hide(axis='index').format(thousands=','))

HM subtype,Unweighted sample n,Unweighted palliative-care n,Weighted hospitalizations,Weighted palliative-care n,Weighted palliative-care %,"95% CI lower, %","95% CI upper, %",Overall p-value
Lymphoma,"8,317","4,148","41,585","20,740",49.870000,48.810000,50.940000,0.002
AML,"4,090","2,068","20,450","10,340",50.560000,49.040000,52.080000,
CML,601,309,"3,005","1,545",51.410000,47.440000,55.370000,
CLL/chronic leukemia,"3,058","1,486","15,290","7,430",48.590000,46.840000,50.350000,
ALL/unspecified acute leukemia,819,362,"4,095","1,810",44.200000,40.840000,47.610000,
Other leukemia,768,374,"3,840","1,870",48.700000,45.190000,52.220000,
Myeloma/plasma-cell neoplasm,"3,876","1,880","19,380","9,400",48.500000,46.940000,50.070000,
MDS,"3,004","1,463","15,020","7,315",48.700000,46.920000,50.480000,
MPN,"2,677","1,239","13,385","6,195",46.280000,44.410000,48.170000,
Total,"27,210","13,329","136,050","66,645",48.990000,48.400000,49.570000,0.002


## Command 19C: Sepsis decedents by HM subtype — adjusted

In [5]:
table_19c_adjusted = pd.read_csv(REPO_ROOT / 'outputs/phase_10/sepsis_decedent_subtype_adjusted.csv', keep_default_na=False)
table_19c_adjusted.columns = ['HM subtype','Adjusted probability, %','95% CI lower, %','95% CI upper, %','Overall p-value']
display(table_19c_adjusted.style.hide(axis='index'))

HM subtype,"Adjusted probability, %","95% CI lower, %","95% CI upper, %",Overall p-value
Lymphoma,50.04,48.99,51.1,0.010
AML,49.77,48.25,51.3,
CML,51.76,47.87,55.66,
CLL/chronic leukemia,46.64,44.89,48.39,
ALL/unspecified acute leukemia,47.08,43.61,50.55,
Other leukemia,48.06,44.57,51.54,
Myeloma/plasma-cell neoplasm,49.82,48.25,51.38,
MDS,47.52,45.76,49.29,
MPN,47.85,45.98,49.73,
Total,—,—,—,0.010


## Missing-outcome review

In [6]:
missing = pd.DataFrame([{'Variable': 'DIED', 'Missing records': summary['died_missing_records']}, {'Variable': 'Total', 'Missing records': summary['died_missing_records']}])
display(missing.style.hide(axis='index').format(thousands=','))

Variable,Missing records
DIED,441
Total,441


## Copy/paste-friendly Markdown

In [7]:
def print_markdown(dataframe, title):
    print(f'## {title}\n')
    headers = list(dataframe.columns); print('| ' + ' | '.join(headers) + ' |'); print('|' + '|'.join(['---'] * len(headers)) + '|')
    for row in dataframe.astype(str).itertuples(index=False, name=None): print('| ' + ' | '.join(value.replace('|', '\\|') for value in row) + ' |')
    print()
print_markdown(table_19a, 'Command 19A')
print_markdown(table_19b, 'Command 19B')
print_markdown(table_19c_raw, 'Command 19C unadjusted')
print_markdown(table_19c_adjusted, 'Command 19C adjusted')

## Command 19A

| Cohort | Unweighted sample n | Unweighted palliative-care n | Weighted hospitalizations | Weighted palliative-care n | Weighted palliative-care % | 95% CI lower, % | 95% CI upper, % | P-value |
|---|---|---|---|---|---|---|---|---|
| Decedents without documented sepsis | 26527 | 15179 | 132635 | 75895 | 57.22 | 56.63 | 57.81 | <0.001 |
| Decedents with documented sepsis | 27210 | 13329 | 136050 | 66645 | 48.99 | 48.4 | 49.57 |  |
| Total decedents | 53737 | 28508 | 268685 | 142540 | 53.05 | 52.64 | 53.46 | <0.001 |

## Command 19B

| HM subtype | Adjusted probability without sepsis, % | Adjusted probability with sepsis, % | Adjusted difference, pp | Difference 95% CI, pp | Subtype p-value | Overall interaction p-value |
|---|---|---|---|---|---|---|
| Lymphoma | 59.08 | 50.57 | -8.52 | -10.06–-6.97 | <0.001 | 0.016 |
| AML | 59.47 | 50.12 | -9.35 | -11.52–-7.18 | <0.001 |  |
| CML | 56.39 | 52.5 | -3.89 | -9.15–1.38 | 0.148 |  |
| CLL/chronic leukemia | 54.75 | 47.49 

## Interpretation

Among in-hospital decedents, documented inpatient palliative-care use was less frequent in hospitalizations with documented sepsis than in those without sepsis. The adjusted sepsis association varied across HM subtypes, and adjusted palliative-care probabilities also differed across subtypes among sepsis decedents. These observational results do not establish causality.